# TCGA-BRCA Indexed Clinical Treatment Layer Review V1

This notebook is review-only. It reads the latest saved patient-treatment-master outputs from disk,
re-exports review TSVs into `05-results`, and does not rerun the integration workflow, mutate prior
raw or source-specific outputs, build treatment arms, perform modeling, or treat indexed treatment
types as full regimen reconstruction.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display


def detect_repo_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Unable to locate the repository root from the notebook path.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


repo_root = detect_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'treatment-prep'
    / 'tcga_brca_patient_treatment_master_v1_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest patient treatment master pointer not found: {latest_pointer_path}'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
run_log_path = repo_root / latest_pointer['run_log_json']
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

master_df = read_tsv(repo_root / latest_pointer['patient_treatment_master_v1_tsv'])
source_audit_df = read_tsv(repo_root / latest_pointer['patient_treatment_master_v1_source_audit_tsv'])
summary_df = read_tsv(repo_root / latest_pointer['patient_treatment_master_v1_summary_tsv'])
missingness_df = read_tsv(repo_root / latest_pointer['patient_treatment_master_v1_missingness_tsv'])

results_root = (
    repo_root
    / '09-trials'
    / '01-tcga-only-source-audited'
    / '05-results'
)
results_root.mkdir(parents=True, exist_ok=True)

master_df.shape, source_audit_df.shape, summary_df.shape, missingness_df.shape


((1097, 79), (1097, 40), (57, 5), (79, 8))

In [2]:
review_master_path = results_root / '140_patient_treatment_master_v1.tsv'
review_source_audit_path = results_root / '141_patient_treatment_master_v1_source_audit.tsv'
review_summary_path = results_root / '142_patient_treatment_master_v1_summary.tsv'
review_missingness_path = results_root / '143_patient_treatment_master_v1_missingness.tsv'

master_df.to_csv(review_master_path, sep='\t', index=False)
source_audit_df.to_csv(review_source_audit_path, sep='\t', index=False)
summary_df.to_csv(review_summary_path, sep='\t', index=False)
missingness_df.to_csv(review_missingness_path, sep='\t', index=False)

print(f'Saved: {review_master_path}')
print(f'Saved: {review_source_audit_path}')
print(f'Saved: {review_summary_path}')
print(f'Saved: {review_missingness_path}')


Saved: d:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\140_patient_treatment_master_v1.tsv
Saved: d:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\141_patient_treatment_master_v1_source_audit.tsv
Saved: d:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\142_patient_treatment_master_v1_summary.tsv
Saved: d:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\143_patient_treatment_master_v1_missingness.tsv


In [3]:
def get_summary_value(metric: str) -> str:
    matches = summary_df.loc[summary_df['summary_metric'] == metric, 'summary_value']
    if matches.empty:
        raise KeyError(f'Missing summary metric: {metric}')
    return matches.iloc[0]


coverage_rows = []
for signal in ['any_treatment', 'druglike', 'radiation', 'non_surgical', 'surgery']:
    coverage_rows.append({
        'signal': signal,
        'local_count': int(get_summary_value(f'{signal}_local_count')),
        'master_count': int(get_summary_value(f'{signal}_master_count')),
        'gain_count': int(get_summary_value(f'{signal}_gain_count')),
        'master_fraction': float(get_summary_value(f'{signal}_master_fraction')),
    })
coverage_df = pd.DataFrame(coverage_rows)

print('=== Latest pointer ===')
display(pd.DataFrame([latest_pointer]))

validation_df = pd.DataFrame(
    [{'check': key, 'value': str(value)} for key, value in run_log.get('validation', {}).items()]
)
print('\n=== Validation ===')
display(validation_df)

counts_df = pd.DataFrame(
    [{'metric': key, 'value': str(value)} for key, value in run_log.get('counts', {}).items()]
)
print('\n=== Key counts ===')
display(counts_df)

print('\n=== Coverage before and after integration ===')
display(coverage_df)

print('\n=== Direct answers ===')
display(summary_df[summary_df['summary_section'] == 'answers'].reset_index(drop=True))


=== Latest pointer ===


,updated_at_utc,patient_treatment_master_v1_run_id,patient_treatment_profile_v1_run_id,indexed_clinical_vs_local_treatment_v1_run_id,treatment_source_audit_run_id,gdc_data_release,gdc_tag,processed_run_directory,audit_run_directory,patient_treatment_master_v1_tsv,patient_treatment_master_v1_source_audit_tsv,patient_treatment_master_v1_summary_tsv,patient_treatment_master_v1_missingness_tsv,run_log_json,patient_treatment_profile_latest_json,indexed_clinical_vs_local_treatment_latest_json
0,2026-04-16T01:46:27Z,20260416T014627Z,20260414T200843Z,20260416T004139Z,20260415T233526Z,"Data Release 45.0 - December 04, 2025",8.3.1,01-data/processed/tcga-brca/treatment-prep/pat...,01-data/audit/tcga-brca/treatment-prep/patient...,01-data/processed/tcga-brca/treatment-prep/pat...,01-data/audit/tcga-brca/treatment-prep/patient...,01-data/audit/tcga-brca/treatment-prep/patient...,01-data/audit/tcga-brca/treatment-prep/patient...,01-data/audit/tcga-brca/treatment-prep/patient...,01-data/audit/tcga-brca/treatment-prep/tcga_br...,01-data/audit/tcga-brca/source/tcga_brca_index...



=== Validation ===


,check,value
0,profile_indexed_run_id_match,True
1,master_row_count_matches_profile,True
2,source_audit_row_count_matches_master,True
3,missingness_row_count_matches_master_columns,True
4,master_unique_barcodes,True
5,indexed_only_cases_excluded_from_master,True
6,profile_fields_preserved_verbatim,True
7,overlap_any_treatment_reconciles,True
8,overlap_druglike_reconciles,True
9,overlap_radiation_reconciles,True



=== Key counts ===


,metric,value
0,patient_treatment_master_v1_row_count,1097
1,indexed_only_case_count_not_integrated,1
2,local_any_treatment_count,819
3,master_any_treatment_count,1097
4,local_druglike_count,780
5,master_druglike_count,1027
6,local_radiation_count,528
7,master_radiation_count,1023
8,local_non_surgical_count,819
9,master_non_surgical_count,1027



=== Coverage before and after integration ===


,signal,local_count,master_count,gain_count,master_fraction
0,any_treatment,819,1097,278,1.0000
1,druglike,780,1027,247,0.9362
2,radiation,528,1023,495,0.9325
3,non_surgical,819,1027,208,0.9362
4,surgery,0,1096,1096,0.9991



=== Direct answers ===


,patient_treatment_master_v1_run_id,summary_section,summary_metric,summary_value,notes
0,20260416T014627Z,answers,how_much_patient_level_treatment_coverage_impr...,any_treatment 819->1097 (+278); druglike 780->...,Coverage gains reflect broad availability and ...
1,20260416T014627Z,answers,how_many_patients_are_supported_by_local_only_...,any_treatment: 0 local_only / 278 indexed_only...,Counts are reported on the integrated 1097-pat...
2,20260416T014627Z,answers,what_still_remains_weak,70 patients remain surgery-only recoveries wit...,"Indexed clinical materially improves coverage,..."
3,20260416T014627Z,answers,whether_dataset_is_strong_enough_for_next_stag...,ready_for_next_treatment_organization_step_wit...,Yes for the next broad treatment-organization ...


In [4]:
support_order = ['local_only', 'indexed_only', 'both', 'neither']
support_columns = {
    'any_treatment': 'master_any_treatment_source_support',
    'druglike': 'master_druglike_source_support',
    'radiation': 'master_radiation_source_support',
    'surgery': 'master_surgery_source_support',
    'non_surgical': 'master_non_surgical_source_support',
    'treatment_timing': 'master_treatment_timing_source_support',
}

provenance_rows = []
for signal, column_name in support_columns.items():
    counts = source_audit_df[column_name].value_counts(dropna=False)
    for support in support_order:
        provenance_rows.append({
            'signal': signal,
            'source_support': support,
            'patient_count': int(counts.get(support, 0)),
        })

provenance_long_df = pd.DataFrame(provenance_rows)
provenance_wide_df = (
    provenance_long_df
    .pivot(index='signal', columns='source_support', values='patient_count')
    .reset_index()
    .rename_axis(columns=None)
)

interpretation_df = (
    source_audit_df.groupby('source_audit_interpretation', as_index=False)
    .size()
    .rename(columns={'size': 'patient_count'})
    .sort_values(['patient_count', 'source_audit_interpretation'], ascending=[False, True])
    .reset_index(drop=True)
)

print('=== Provenance breakdowns ===')
display(provenance_wide_df)

print('\n=== Source-audit interpretation breakdown ===')
display(interpretation_df)


=== Provenance breakdowns ===


,signal,both,indexed_only,local_only,neither
0,any_treatment,819,278,0,0
1,druglike,780,247,0,70
2,non_surgical,819,208,0,70
3,radiation,528,495,0,74
4,surgery,0,1096,0,1
5,treatment_timing,808,6,1,282



=== Source-audit interpretation breakdown ===


,source_audit_interpretation,patient_count
0,indexed_fills_local_gap,534
1,local_and_indexed_concordant,493
2,indexed_surgery_only_recovery,70


In [5]:
focus_columns = [
    'bcr_patient_barcode',
    'indexed_case_id',
    'local_profile_has_any_drug_row',
    'local_profile_has_any_radiation_row',
    'master_druglike_source_support',
    'master_radiation_source_support',
    'master_non_surgical_source_support',
    'indexed_surgery_only_flag',
    'indexed_incremental_value_class',
    'indexed_treatment_type_values_json',
    'indexed_therapeutic_agents_values_json',
    'indexed_regimen_line_values_json',
]

indexed_surgery_only_df = source_audit_df[
    source_audit_df['indexed_surgery_only_flag'] == 'yes'
][focus_columns].reset_index(drop=True)

no_master_druglike_df = source_audit_df[
    source_audit_df['master_druglike_source_support'] == 'neither'
][focus_columns].reset_index(drop=True)

no_master_radiation_df = source_audit_df[
    source_audit_df['master_radiation_source_support'] == 'neither'
][focus_columns].reset_index(drop=True)

remaining_missingness_df = missingness_df[
    missingness_df['missing_like_fraction'] != '0.000000'
].reset_index(drop=True)

print('=== Indexed surgery-only recovery subset ===')
display(indexed_surgery_only_df)

print('\n=== No master druglike evidence after integration ===')
display(no_master_druglike_df)

print('\n=== No master radiation evidence after integration ===')
display(no_master_radiation_df)

print('\n=== Missingness review snapshot ===')
display(remaining_missingness_df)


=== Indexed surgery-only recovery subset ===


,bcr_patient_barcode,indexed_case_id,local_profile_has_any_drug_row,local_profile_has_any_radiation_row,master_druglike_source_support,master_radiation_source_support,master_non_surgical_source_support,indexed_surgery_only_flag,indexed_incremental_value_class,indexed_treatment_type_values_json,indexed_therapeutic_agents_values_json,indexed_regimen_line_values_json
0,TCGA-A1-A0SB,0045349c-69d9-4306-a403-c9c1fa836644,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
1,TCGA-A1-A0SD,c462e422-eb8d-4daf-9897-2a9c6cbd783a,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
2,TCGA-A1-A0SE,c397e6c2-ba3a-4f5e-a258-c140611192fe,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
3,TCGA-A1-A0SI,16368c32-2118-4fcf-8693-6c89995e49d8,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
4,TCGA-A1-A0SJ,a2db9dd1-44d5-48b5-817c-a21a85fadb21,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...
65,TCGA-C8-A12K,6b923e30-2ddc-407f-91b1-c202f1373fbb,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
66,TCGA-C8-A12T,71229981-97c2-4ae1-a130-fe5de336b37d,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
67,TCGA-C8-A133,b8b6cebc-26dc-43ae-a94a-20da6018f7ae,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
68,TCGA-C8-A1HJ,a855c228-a263-44df-87a0-cbc32187e3f5,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]



=== No master druglike evidence after integration ===


,bcr_patient_barcode,indexed_case_id,local_profile_has_any_drug_row,local_profile_has_any_radiation_row,master_druglike_source_support,master_radiation_source_support,master_non_surgical_source_support,indexed_surgery_only_flag,indexed_incremental_value_class,indexed_treatment_type_values_json,indexed_therapeutic_agents_values_json,indexed_regimen_line_values_json
0,TCGA-A1-A0SB,0045349c-69d9-4306-a403-c9c1fa836644,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
1,TCGA-A1-A0SD,c462e422-eb8d-4daf-9897-2a9c6cbd783a,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
2,TCGA-A1-A0SE,c397e6c2-ba3a-4f5e-a258-c140611192fe,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
3,TCGA-A1-A0SI,16368c32-2118-4fcf-8693-6c89995e49d8,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
4,TCGA-A1-A0SJ,a2db9dd1-44d5-48b5-817c-a21a85fadb21,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...
65,TCGA-C8-A12K,6b923e30-2ddc-407f-91b1-c202f1373fbb,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
66,TCGA-C8-A12T,71229981-97c2-4ae1-a130-fe5de336b37d,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
67,TCGA-C8-A133,b8b6cebc-26dc-43ae-a94a-20da6018f7ae,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
68,TCGA-C8-A1HJ,a855c228-a263-44df-87a0-cbc32187e3f5,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]



=== No master radiation evidence after integration ===


,bcr_patient_barcode,indexed_case_id,local_profile_has_any_drug_row,local_profile_has_any_radiation_row,master_druglike_source_support,master_radiation_source_support,master_non_surgical_source_support,indexed_surgery_only_flag,indexed_incremental_value_class,indexed_treatment_type_values_json,indexed_therapeutic_agents_values_json,indexed_regimen_line_values_json
0,TCGA-A1-A0SB,0045349c-69d9-4306-a403-c9c1fa836644,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
1,TCGA-A1-A0SD,c462e422-eb8d-4daf-9897-2a9c6cbd783a,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
2,TCGA-A1-A0SE,c397e6c2-ba3a-4f5e-a258-c140611192fe,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
3,TCGA-A1-A0SF,4b620966-b4c7-4615-bf58-ca9d3226b9bd,yes,no,both,neither,both,no,coarse_only,"[""Hormone Therapy"", ""Pharmaceutical Therapy, N...","[""Tamoxifen"", ""Anastrozole"", ""Carboplatin"", ""P...","[""TC""]"
4,TCGA-A1-A0SG,0ed9ad26-7fdc-4d75-93f9-d029eee94993,yes,no,both,neither,both,no,coarse_only,"[""Chemotherapy"", ""Surgery, NOS""]",[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...
69,TCGA-C8-A12K,6b923e30-2ddc-407f-91b1-c202f1373fbb,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
70,TCGA-C8-A12T,71229981-97c2-4ae1-a130-fe5de336b37d,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
71,TCGA-C8-A133,b8b6cebc-26dc-43ae-a94a-20da6018f7ae,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]
72,TCGA-C8-A1HJ,a855c228-a263-44df-87a0-cbc32187e3f5,no,no,neither,neither,neither,yes,coarse_only,"[""Surgery, NOS""]",[],[]



=== Missingness review snapshot ===


,field_name,field_group,row_count,non_missing_count,missing_like_count,missing_like_fraction,distinct_non_missing_count,notes
0,drug_therapy_type_values_json,local_profile,1097,780,317,0.288970,143,json_or_text_set_field; [] counted as missing-...
1,dominant_therapy_type_if_any,local_profile,1097,729,368,0.335460,5,local_profile
2,therapy_type_counts_json,local_profile,1097,778,319,0.290793,111,json_or_text_set_field; [] counted as missing-...
3,regimen_context_values_json,local_profile,1097,467,630,0.574294,14,json_or_text_set_field; [] counted as missing-...
4,drug_name_values_json,local_profile,1097,780,317,0.288970,526,json_or_text_set_field; [] counted as missing-...
5,earliest_drug_start_days,local_profile,1097,759,338,0.308113,206,local_profile
6,latest_drug_end_days,local_profile,1097,598,499,0.454877,360,local_profile
7,earliest_radiation_start_days,local_profile,1097,506,591,0.538742,248,local_profile
8,latest_radiation_end_days,local_profile,1097,507,590,0.537830,263,local_profile
9,indexed_treatment_intent_type_values_json,indexed_raw,1097,1096,1,0.000912,18,json_or_text_set_field; [] counted as missing-...
